# Mini-TP 6 · Tu modelo en el Data Lake (individual)

**Operaciones de Aprendizaje Automático II · CEIA – FIUBA**

## Consigna
Sube el artefacto de **tu propio modelo** a un Data Lake (MinIO/S3) y **sírvelo cargándolo
desde allí**, no desde disco. Organiza el bucket con **zonas** y **versión**.

### Qué se entrega
1. El bucket con zonas (`raw`, `curated`, `models`) y tu modelo en `models/v1/…`.
2. Una función que **carga el modelo desde el lake** y predice.
3. (Opcional) El experimento registrado con **MLflow apuntando a MinIO/S3**.
4. Una reflexión breve: ¿por qué servir desde el lake y no hornear el modelo en la imagen?

### Se evalúa
- Que **corra de punta a punta**.
- Uso correcto de la API S3 (`boto3`): bucket, zonas por prefijo, versión del modelo.
- Que el modelo se **cargue desde el lake** para servir.
- (Opcional) Integración con MLflow.

> Antes de empezar, levanta MinIO con Docker (ver la *Guía de la práctica* o el tutorial
> `Practica/datalake_tutorial.ipynb`) y instala las dependencias:
> `uv add boto3 pandas pyarrow scikit-learn mlflow joblib`


## 0. Configuración
Uso el **MinIO del stack de MLOps1_final** (ya levantado con `docker compose`), que expone la
API S3 en `localhost:9000`. Por eso las credenciales son las de ese stack (`minioadmin`) y no
las del tutorial (`minio`). El bucket es nuevo y propio de este Mini-TP, así no se mezcla con los
buckets que usa el TP integrador (`raw-data`, `mlflow`).

El modelo es el mismo del TP integrador: **XGBoost `xgb_best`** que predice el precio de autos
usados (dataset Car Dekho). Se reusa el preprocesamiento de `MLOps1_final/common` sin copiarlo.

In [2]:
import os, sys
os.environ.setdefault("AWS_ACCESS_KEY_ID", "minioadmin")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "minioadmin123")
os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")

ENDPOINT = "http://localhost:9000"
BUCKET = "datalake-tp6"      # bucket propio del Mini-TP 6
print("Endpoint:", ENDPOINT, "| Bucket:", BUCKET)

# Reusar el preprocesamiento del TP integrador (MLOps1_final/common)
TP_MLOPS1 = os.path.abspath(os.path.join("..", "TP_Final", "MLOps1_final"))
if TP_MLOPS1 not in sys.path:
    sys.path.insert(0, TP_MLOPS1)
DATASET_CSV = os.path.join(TP_MLOPS1, "dataset", "Car details v3.csv")
print("Dataset encontrado:", os.path.exists(DATASET_CSV))

Endpoint: http://localhost:9000 | Bucket: datalake-tp6
Dataset encontrado: True


In [3]:
import boto3
from botocore.client import Config

# Un único cliente S3: el mismo código sirve para MinIO local y para S3 en la nube
# (en la nube solo se quita endpoint_url y se usan credenciales de AWS).
s3 = boto3.client(
    "s3",
    endpoint_url=ENDPOINT,
    config=Config(signature_version="s3v4"),
)
print("Cliente S3 listo. Buckets existentes:", [b["Name"] for b in s3.list_buckets()["Buckets"]])

Cliente S3 listo. Buckets existentes: ['datalake-tp6', 'mlflow-artifacts', 'raw-data']


## 1. Crear el bucket y las zonas


In [4]:
def crear_bucket(bucket):
    existentes = [b["Name"] for b in s3.list_buckets()["Buckets"]]
    if bucket in existentes:
        print(f"El bucket '{bucket}' ya existe.")
        return
    s3.create_bucket(Bucket=bucket)
    print(f"Bucket '{bucket}' creado.")

crear_bucket(BUCKET)

# En S3 no hay carpetas: las zonas son prefijos. Se crean subiendo un '.keep' vacío.
ZONAS = ["raw", "curated", "models"]
for z in ZONAS:
    s3.put_object(Bucket=BUCKET, Key=f"{z}/.keep", Body=b"")
print("Zonas inicializadas:", ZONAS)

El bucket 'datalake-tp6' ya existe.
Zonas inicializadas: ['raw', 'curated', 'models']


## 2. Entrenar tu modelo
Entreno **mi modelo** del curso: XGBoost con los mismos pasos y los mismos hiperparámetros que
usa el DAG de Airflow del TP integrador (`common/preprocessing.py` y `common/constants.py`).

- `df_raw`: el dataset tal cual viene (va a la zona `raw`).
- `df_curated`: el dataset limpio, con las features ya calculadas y el target (va a `curated`).
- `metadata`: lo que el modelo necesita para preprocesar un auto nuevo al predecir
  (lista de marcas principales y orden de las columnas). Se sube **junto al modelo**.

In [6]:
import io, json, joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from xgboost import XGBRegressor

from common.constants import XGB_PARAMS, TRAIN_TEST_SIZE, TRAIN_TEST_SPLIT_SEED
from common.preprocessing import (
    extraer_columnas_tecnicas, drop_missing_technical_rows, drop_km_outliers,
    compute_top_brands, encode_categoricals, get_all_feature_columns,
    align_columns, build_feature_matrix,
)

# Dataset crudo
df_raw = pd.read_csv(DATASET_CSV)
df = df_raw.drop_duplicates().reset_index(drop=True)

# Split estratificado por rango de precio (igual que en el TP integrador)
price_bins = pd.qcut(df["selling_price"], q=5,
                     labels=["muy_barato", "barato", "intermedio", "caro", "muy_caro"])
df_train, df_test = train_test_split(df, test_size=TRAIN_TEST_SIZE,
                                     stratify=price_bins, random_state=TRAIN_TEST_SPLIT_SEED)

# Feature engineering
df_train = drop_km_outliers(drop_missing_technical_rows(extraer_columnas_tecnicas(df_train)))
df_test = drop_missing_technical_rows(extraer_columnas_tecnicas(df_test))
top_brands = compute_top_brands(df_train)
df_train_enc, df_test_enc = encode_categoricals(df_train, df_test, top_brands)
all_features = get_all_feature_columns(df_train_enc)
df_train_enc, df_test_enc = align_columns(df_train_enc, df_test_enc, all_features)

X_train = build_feature_matrix(df_train_enc, all_features)
X_test = build_feature_matrix(df_test_enc, all_features)
y_train = np.log1p(df_train_enc["selling_price"])   # el modelo predice log(precio)
y_test = df_test_enc["selling_price"]

# Entrenamiento
modelo = XGBRegressor(**XGB_PARAMS)
modelo.fit(X_train, y_train)

pred_test = np.expm1(modelo.predict(X_test))
r2 = r2_score(y_test, pred_test)
mae = mean_absolute_error(y_test, pred_test)
print(f"Modelo entrenado. R2 test = {r2:.4f} | MAE test = {mae:,.0f}")

# Dataset curado: features + target, listo para entrenar (train y test juntos, marcados)
df_curated = pd.concat([
    X_train.assign(selling_price=df_train_enc["selling_price"].values, split="train"),
    X_test.assign(selling_price=y_test.values, split="test"),
], ignore_index=True)

# Metadata que el servicio necesita para preprocesar al predecir
metadata = {
    "model_name": "xgb_best",
    "version": "v1",
    "framework": "xgboost",
    "target": "log1p(selling_price)",
    "top_brands": top_brands,
    "all_features": all_features,
    "metrics": {"r2_test": round(float(r2), 4), "mae_test": round(float(mae), 2)},
}
print("Filas raw:", len(df_raw), "| filas curated:", len(df_curated), "| n_features:", len(all_features))

Modelo entrenado. R2 test = 0.9372 | MAE test = 72,408
Filas raw: 8128 | filas curated: 6715 | n_features: 23


## 3. Subir dataset y modelo al lake


In [7]:
DATASET = "car_dekho"
VERSION = "v1"
KEY_MODELO = f"models/{VERSION}/xgb_best.pkl"
KEY_META = f"models/{VERSION}/metadata.json"

# a) Dataset CRUDO a raw (CSV, tal cual vino)
buf_csv = io.StringIO(); df_raw.to_csv(buf_csv, index=False)
s3.put_object(Bucket=BUCKET, Key=f"raw/{DATASET}/car_details_v3.csv",
              Body=buf_csv.getvalue().encode("utf-8"))

# b) Dataset CURADO a curated (Parquet: columnar, más liviano y con tipos)
buf_pq = io.BytesIO(); df_curated.to_parquet(buf_pq, index=False)
s3.put_object(Bucket=BUCKET, Key=f"curated/{DATASET}/car_features.parquet", Body=buf_pq.getvalue())

# c) MODELO versionado a models/v1 (joblib -> bytes -> put_object), con su metadata al lado
buf_model = io.BytesIO(); joblib.dump(modelo, buf_model)
s3.put_object(Bucket=BUCKET, Key=KEY_MODELO, Body=buf_model.getvalue())
s3.put_object(Bucket=BUCKET, Key=KEY_META,
              Body=json.dumps(metadata, indent=2).encode("utf-8"),
              ContentType="application/json")

# d) Puntero a la versión en producción: el servicio lee este archivo para saber qué versión cargar
s3.put_object(Bucket=BUCKET, Key="models/PRODUCTION", Body=VERSION.encode("utf-8"))

def listar(bucket, prefix=""):
    resp = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
    for obj in resp.get("Contents", []):
        if obj["Key"].endswith("/.keep"):
            continue
        print(f"  {obj['Key']:60s} {obj['Size']:>10,} bytes")

print(f"Contenido de s3://{BUCKET}/")
listar(BUCKET)

Contenido de s3://datalake-tp6/
  curated/car_dekho/car_features.parquet                           93,873 bytes
  mlflow/123b3cdabc3443edbb2fa0fa7ffc5589/artifacts/model/xgb_best.pkl  1,456,989 bytes
  mlflow/d8050266f65149bcb624553a07ac99c7/artifacts/model/xgb_best.pkl  1,456,989 bytes
  models/PRODUCTION                                                     2 bytes
  models/v1/metadata.json                                             888 bytes
  models/v1/xgb_best.pkl                                        1,456,989 bytes
  raw/car_dekho/car_details_v3.csv                              1,057,451 bytes


## 4. Servir el modelo desde el lake
El corazón del Mini-TP: **cargar el modelo desde el lake** (no desde disco) y predecir.


In [8]:
from common.schemas import CarRawInput
from common.preprocessing import build_inference_features

# Borro el modelo de memoria para demostrar que lo que sigue viene del lake y no del notebook.
del modelo

def cargar_modelo_desde_lake(bucket, key):
    """Descarga el modelo del lake y lo reconstruye en memoria (nunca toca el disco)."""
    obj = s3.get_object(Bucket=bucket, Key=key)
    return joblib.load(io.BytesIO(obj["Body"].read()))

def cargar_metadata_desde_lake(bucket, key):
    obj = s3.get_object(Bucket=bucket, Key=key)
    return json.loads(obj["Body"].read())

def version_en_produccion(bucket):
    obj = s3.get_object(Bucket=bucket, Key="models/PRODUCTION")
    return obj["Body"].read().decode("utf-8").strip()

# Lo que haría el servicio al arrancar: ver qué versión está en producción y bajarla del lake
version = version_en_produccion(BUCKET)
modelo_lake = cargar_modelo_desde_lake(BUCKET, f"models/{version}/xgb_best.pkl")
meta_lake = cargar_metadata_desde_lake(BUCKET, f"models/{version}/metadata.json")
print(f"Modelo cargado desde s3://{BUCKET}/models/{version}/ ->", type(modelo_lake).__name__)

def predecir_precio(auto: dict) -> float:
    car = CarRawInput(**auto)   # misma validación que la API REST
    fila = build_inference_features(car, meta_lake["top_brands"], meta_lake["all_features"])
    return float(np.expm1(modelo_lake.predict(fila)[0]))

# Predicción de ejemplo (el primer auto del dataset, precio real 450.000)
auto = {
    "name": "Maruti Swift Dzire VDI", "year": 2014, "km_driven": 145500,
    "fuel": "Diesel", "seller_type": "Individual", "transmission": "Manual",
    "owner": "First Owner", "mileage": "23.4 kmpl", "engine": "1248 CC",
    "max_power": "74 bhp", "torque": "190Nm@ 2000rpm", "seats": 5,
}
print(f"Precio predicho: {predecir_precio(auto):,.0f}  (precio real: 450,000)")

# Chequeo: el modelo del lake predice igual que el entrenado en el notebook
pred_lake = np.expm1(modelo_lake.predict(X_test))
print("Mismas predicciones que antes de subirlo:", np.allclose(pred_lake, pred_test))

Modelo cargado desde s3://datalake-tp6/models/v1/ -> XGBRegressor
Precio predicho: 455,294  (precio real: 450,000)
Mismas predicciones que antes de subirlo: True


## 5. (Opcional) MLflow apuntando a MinIO/S3


In [9]:
import tempfile
os.environ["MLFLOW_S3_ENDPOINT_URL"] = ENDPOINT     # MinIO como almacenamiento S3 de MLflow
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"

import mlflow
from mlflow.tracking import MlflowClient

# Tracking local en SQLite, artefactos en el lake (s3://datalake-tp6/mlflow)
mlflow.set_tracking_uri("sqlite:///mlflow_tp6.db")
client = MlflowClient()
NOMBRE_EXP = "mini-tp6-datalake"
exp = client.get_experiment_by_name(NOMBRE_EXP)
exp_id = exp.experiment_id if exp else client.create_experiment(
    NOMBRE_EXP, artifact_location=f"s3://{BUCKET}/mlflow")
mlflow.set_experiment(experiment_id=exp_id)

with mlflow.start_run(run_name="xgb_best-v1") as run:
    mlflow.log_params({k: v for k, v in XGB_PARAMS.items() if k != "n_jobs"})
    mlflow.log_metrics(metadata["metrics"])
    with tempfile.TemporaryDirectory() as tmp:
        ruta = os.path.join(tmp, "xgb_best.pkl")
        joblib.dump(modelo_lake, ruta)
        mlflow.log_artifact(ruta, artifact_path="model")
    run_id = run.info.run_id
print("Run registrado:", run_id)

print("\nArtefactos de MLflow en el lake:")
listar(BUCKET, prefix=f"mlflow/{run_id}")

Run registrado: c1c2400b683140249a1aef58955eed14

Artefactos de MLflow en el lake:
  mlflow/c1c2400b683140249a1aef58955eed14/artifacts/model/xgb_best.pkl  1,456,989 bytes


## 6. Reflexión (texto)

Responde en unas pocas líneas:

1. ¿Por qué conviene que el servicio **cargue el modelo desde el lake** en vez de
   incluirlo dentro de la imagen del contenedor?
2. ¿Cómo versionarías un modelo nuevo sin romper el que ya está en producción?
3. ¿Qué zona del lake usarías para guardar las predicciones que sirve tu API por
   GraphQL, y por qué?

**1. Cargar desde el lake y no dentro de la imagen.**
Si el modelo está dentro de la imagen, cada reentrenamiento obliga a construir y desplegar una
imagen nueva, aunque el código no haya cambiado. Cargándolo desde el lake, el código y el modelo
se actualizan por separado: Airflow entrena, sube `models/v2/` y el servicio solo se reinicia.
Además, la imagen queda más chica, y las cuatro capas de mi TP (REST, GraphQL, gRPC y streaming)
leen **el mismo archivo**, sin riesgo de que cada una tenga una copia distinta.

**2. Versionar sin romper producción.**
Cada versión va en su propia carpeta (`models/v1/`, `models/v2/`, ...) y **nunca se pisa** una
versión ya subida. El archivo `models/PRODUCTION` dice qué versión usa el servicio. Para pasar a
v2: subo `models/v2/`, la pruebo, y recién ahí cambio `PRODUCTION` de `v1` a `v2`. Si algo falla,
vuelvo a escribir `v1` y listo: volver atrás es cambiar un texto. Es la misma idea que el stage
`Production` del Model Registry de MLflow que uso en el TP integrador.

**3. Zona para las predicciones que sirve GraphQL.**
En **`raw`**, separadas por fecha (por ejemplo `raw/predicciones/fecha=2026-10-01/...`), guardando
la entrada, el precio predicho y la versión del modelo. Van a `raw` porque son datos nuevos tal
como llegaron, todavía sin revisar ni limpiar. Después, un proceso las puede limpiar y pasar a
`curated` para medir el drift o para reentrenar el modelo cuando se conozca el precio real de venta.